# Probe — run and commit this first

Answers three things nothing else can: the hidden test-set size, whether the
test split ships ground truth, and the zarr version/layout in this image.
Also writes a format-valid `submission.csv` so the submit button unlocks.

In [ ]:
import csv, os
from pathlib import Path

roots = [Path("/kaggle/input") / d for d in os.listdir("/kaggle/input")]
print("inputs:", [r.name for r in roots])
for r in roots:
    for sub in sorted(p for p in r.rglob("*") if p.is_dir() and p.suffix != ".zarr"):
        n = len(list(sub.glob("*.zarr")))
        if n:
            print(f"  {sub.relative_to(r)}: {n} .zarr")

In [ ]:
test_dir = next(p for r in roots for p in r.rglob("*")
                if p.is_dir() and list(p.glob("*.zarr")) and "test" in p.name.lower())
names = sorted(p.stem for p in test_dir.glob("*.zarr"))

print("dir:", test_dir)
print("TEST SET SIZE:", len(names))
print("first 5:", names[:5])
print("prefixes:", sorted({n.split("_")[0] for n in names}))
print("has .geff:", len(list(test_dir.glob("*.geff"))))

In [ ]:
import zarr
print("zarr", zarr.__version__)

g = zarr.open_group(str(test_dir / f"{names[0]}.zarr"), mode="r")
print("shape:", g["0"].shape, g["0"].dtype)
print("chunks:", g["0"].chunks)
print("attr keys:", list(g.attrs))
print("quantiles:", g.attrs["image_statistics"]["quantiles"])

In [ ]:
# Minimal format-valid submission: one node per dataset, every dataset present.
with open("submission.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
                "source_id", "target_id"])
    for i, n in enumerate(names):
        w.writerow([i, n, "node", i + 1, 0, 32, 128, 128, -1, -1])

print(f"wrote {len(names)} rows")
print(open("submission.csv").read()[:300])